In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------
# 1. Paths
# -----------------------------
data_file = r"C:\Users\dsp7\Box\MY RESEARCH\India Sentiment Project\GKG_Data\india_with_article_text_NLP.csv"
output_folder = os.path.dirname(data_file)
cleaned_file = os.path.join(output_folder, "news_data_2024_V2.csv")
descriptives_folder = os.path.join(output_folder, "descriptives")
plots_folder = os.path.join(descriptives_folder, "plots")

os.makedirs(descriptives_folder, exist_ok=True)
os.makedirs(plots_folder, exist_ok=True)

# -----------------------------
# 2. Load Data
# -----------------------------
df = pd.read_csv(data_file)

# -----------------------------
# 3. Clean District/State/Country
# -----------------------------
def clean_location(row):
    full = str(row['District']).strip()
    state = str(row['State']).strip() if pd.notna(row['State']) else np.nan
    # default values
    district_clean, state_clean, country_clean = np.nan, np.nan, 'India'

    parts = [p.strip() for p in full.split(',')]
    if len(parts) == 1:
        if parts[0].lower() in ['india', 'hindustan']:
            country_clean = 'India'
        elif 'river' in parts[0].lower():
            district_clean = np.nan
        else:
            district_clean = parts[0]
    elif len(parts) >= 2:
        # Assume format: District, State, Country
        if parts[0].lower() == parts[1].lower():
            district_clean = np.nan
            state_clean = parts[0]
        else:
            district_clean = parts[0]
            state_clean = parts[1]

        # special case: Telangana correction
        if district_clean is not np.nan and state_clean is not np.nan:
            if str(district_clean).lower() == 'telangana' and str(state_clean).lower() == 'andhra pradesh':
                state_clean = 'Telangana'
                district_clean = np.nan

        # River check
        if district_clean is not np.nan:
            if 'river' in str(district_clean).lower():
                district_clean = np.nan

        # Hindustan case
        if district_clean is not np.nan:
            if str(district_clean).lower() == 'hindustan':
                district_clean = np.nan
                state_clean = np.nan
                country_clean = 'India'
                # --- Standardize State Names ---
        if state_clean is not np.nan:
            state_lower = str(state_clean).lower()
            if state_lower == 'uttaranchal':
                state_clean = 'Uttarakhand'
            elif state_lower in ['jammu and kashmir', 'jammu and kashmir'.lower()]:
                state_clean = 'Jammu & Kashmir'
            elif state_lower == 'orissa':
                state_clean = 'Odisha'

        

    return pd.Series([district_clean, state_clean, country_clean])


df[['District_Clean', 'State_Clean', 'Country']] = df.apply(clean_location, axis=1)

# -----------------------------
# 4. Parse Date
# -----------------------------
df['Date'] = pd.to_datetime(df['DATE'].astype(str), format='%Y%m%d', errors='coerce')
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Month_Name'] = df['Date'].dt.strftime('%b')

months_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

# -----------------------------
# 5. Save Cleaned CSV
# -----------------------------
df.to_csv(cleaned_file, index=False)
